# field vegetation × annual K2 supervised dataset

This notebook builds supervised Sentinel-2 K2 datasets for **two vegetation response resolutions in parallel**:

1. **Parent clusters** — broad functional-group vegetation states (`parent_cluster`)
2. **Subclusters / final states** — nested vegetation states (`state_id`)

Both use the identical IA45 optical predictor matrix so downstream model performance can be compared directly without committing to one ecological resolution in advance.

## Coordinate authority

All plot positions come from:

`C:\NCA_DATA\Analysis\2016thru2026_plots_master.csv`

Coordinates are NAD83 / UTM Zone 11N.

## Analysis-year rule

A vegetation year enters this supervised experiment **only when its finalized annual K2 raster exists**.

Therefore 2026 vegetation is deliberately excluded from the current coordinate join and supervised dataset because no 2026 annual K2 raster exists yet. It can be added automatically once that raster is present.

## Optical predictors

Primary predictors are IA45:

- 15 intercepts
- 30 amplitudes

The full sampled 91-band values are retained for QA.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
from IPython.display import display

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

# =============================================================================
# PATHS
# =============================================================================

VEG_STATE_CSV = Path(
    r"C:\NCA_DATA\Vegetation Data\Clustering_Veg\Derived"
    r"\NCA_vegetation_states_final.csv"
)

MASTER_POINT_CSV = Path(
    r"C:\NCA_DATA\Analysis\2016thru2026_plots_master.csv"
)

HARMONIC_ROOT = Path(r"A:\NCA_DATA\S2_Harmonics")

OUTPUT_DIR = Path(
    r"A:\NCA_DATA\Validation\Supervised_State_Classification"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ALL = OUTPUT_DIR / "FieldVeg_K2_AllSampledBands.csv"
OUT_PARENT = OUTPUT_DIR / "FieldVeg_K2_IA45_ParentClusters.csv"
OUT_SUBCLUSTER = OUTPUT_DIR / "FieldVeg_K2_IA45_Subclusters.csv"
OUT_AUDIT = OUTPUT_DIR / "FieldVeg_K2_ExtractionAudit.csv"
OUT_YEAR_SUMMARY = OUTPUT_DIR / "FieldVeg_K2_YearSummary.csv"

EXPECTED_CRS = "EPSG:26911"
EXPECTED_IA_FEATURES = 45

print("Vegetation states:", VEG_STATE_CSV)
print("Position master:   ", MASTER_POINT_CSV)
print("Harmonic root:     ", HARMONIC_ROOT)
print("Output directory:  ", OUTPUT_DIR)


Vegetation states: C:\NCA_DATA\Vegetation Data\Clustering_Veg\Derived\NCA_vegetation_states_final.csv
Position master:    C:\NCA_DATA\Analysis\2016thru2026_plots_master.csv
Harmonic root:      A:\NCA_DATA\S2_Harmonics
Output directory:   A:\NCA_DATA\Validation\Supervised_State_Classification


In [2]:
# =============================================================================
# HELPERS
# =============================================================================

def clean_text(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return None
    return text


def normalize_plot_id_value(value):
    text = clean_text(value)
    if text is None:
        return None
    return re.sub(r"\.0$", "", text)


def normalize_plot_id(series):
    return series.map(normalize_plot_id_value)


def find_col(df, candidates, required=True):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in lookup:
            return lookup[key]
    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns include: {list(df.columns)[:50]}"
        )
    return None


def annual_k2_path(year):
    year = int(year)
    return (
        HARMONIC_ROOT
        / str(year)
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )


def get_band_names(src):
    names = list(src.descriptions)

    unnamed = [
        i + 1
        for i, name in enumerate(names)
        if name is None or str(name).strip() == ""
    ]
    if unnamed:
        raise RuntimeError(
            f"{src.name} contains unnamed bands at positions {unnamed[:20]}"
        )

    names = [str(name) for name in names]

    if len(set(names)) != len(names):
        s = pd.Series(names)
        dupes = s.loc[s.duplicated(keep=False)].tolist()
        raise RuntimeError(
            f"{src.name} contains duplicate band descriptions: {dupes[:20]}"
        )

    return names


def identify_ia_bands(band_names):
    intercept = [
        b for b in band_names
        if re.search(r"^(constant_|intercept_)", b, flags=re.IGNORECASE)
    ]

    amplitude = [
        b for b in band_names
        if re.search(r"^(amp|amplitude)", b, flags=re.IGNORECASE)
    ]

    ia = intercept + amplitude

    print(f"Intercept bands: {len(intercept)}")
    print(f"Amplitude bands: {len(amplitude)}")
    print(f"IA total:        {len(ia)}")

    if len(ia) != EXPECTED_IA_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_IA_FEATURES} IA bands, found {len(ia)}.\n"
            f"Intercept example: {intercept[:8]}\n"
            f"Amplitude example: {amplitude[:8]}"
        )

    return intercept, amplitude, ia


def sample_year_raster(raster_path, point_df):
    # Points outside raster bounds are retained with all raster bands = NaN.
    point_df = point_df.copy().reset_index(drop=True)

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise RuntimeError(f"{raster_path} has no CRS.")

        crs_string = src.crs.to_string()

        if crs_string.upper() != EXPECTED_CRS.upper():
            raise RuntimeError(
                f"{raster_path.name}: CRS is {crs_string}, "
                f"expected {EXPECTED_CRS}."
            )

        band_names = get_band_names(src)

        left, bottom, right, top = src.bounds

        point_df["in_raster_bounds"] = (
            point_df["Easting"].between(left, right, inclusive="both")
            & point_df["Northing"].between(bottom, top, inclusive="both")
        )

        sampled = pd.DataFrame(
            np.nan,
            index=np.arange(len(point_df)),
            columns=band_names,
            dtype=np.float32,
        )

        inside_idx = point_df.index[
            point_df["in_raster_bounds"]
        ].tolist()

        if inside_idx:
            inside_coords = list(
                zip(
                    point_df.loc[inside_idx, "Easting"].astype(float),
                    point_df.loc[inside_idx, "Northing"].astype(float),
                )
            )

            values = np.asarray(
                list(
                    src.sample(
                        inside_coords,
                        indexes=list(range(1, src.count + 1)),
                        masked=False,
                    )
                ),
                dtype=np.float32,
            )

            inside_sampled = pd.DataFrame(
                values,
                columns=band_names,
                index=inside_idx,
            )

            if src.nodata is not None:
                inside_sampled = inside_sampled.replace(src.nodata, np.nan)

            inside_sampled = inside_sampled.replace(
                [np.inf, -np.inf],
                np.nan,
            )

            sampled.loc[inside_idx, band_names] = inside_sampled

    sampled.insert(0, "PlotID", point_df["PlotID"].to_numpy())
    sampled.insert(1, "Year", point_df["Year"].to_numpy())
    sampled.insert(2, "Easting", point_df["Easting"].to_numpy())
    sampled.insert(3, "Northing", point_df["Northing"].to_numpy())
    sampled.insert(
        4,
        "in_raster_bounds",
        point_df["in_raster_bounds"].to_numpy(),
    )

    return sampled


## 1. Load the final vegetation-state checkpoint


In [3]:
# =============================================================================
# VEGETATION TARGET
# =============================================================================

if not VEG_STATE_CSV.exists():
    raise FileNotFoundError(
        f"Final vegetation-state checkpoint not found:\n{VEG_STATE_CSV}"
    )

veg = pd.read_csv(VEG_STATE_CSV, low_memory=False)
veg = veg.drop(columns=["Unnamed: 0"], errors="ignore")

# Historical vegetation tables used X as a 1-based export/index field.
if "X" in veg.columns:
    veg = veg.rename(columns={"X": "veg_row_index"})

if "PlotID" not in veg.columns:
    if "Plot" in veg.columns:
        veg = veg.rename(columns={"Plot": "PlotID"})
    else:
        raise KeyError(
            "Vegetation-state table contains neither PlotID nor Plot."
        )

if "Year" not in veg.columns:
    raise KeyError(
        "Vegetation-state table does not contain Year."
    )

veg["PlotID"] = normalize_plot_id(veg["PlotID"])
veg["Year"] = pd.to_numeric(veg["Year"], errors="raise").astype(int)

required_target_cols = [
    "parent_cluster",
    "max_membership",
    "membership_margin",
    *[f"membership_{i}" for i in range(1, 9)],
    "raw_subcluster",
    "state_id",
    "state_label",
    "state_type",
    "is_reference",
]

missing_target_cols = [
    c for c in required_target_cols
    if c not in veg.columns
]

if missing_target_cols:
    raise KeyError(
        "Final vegetation-state checkpoint is missing expected target fields: "
        f"{missing_target_cols}"
    )

veg_dup = veg.duplicated(["PlotID", "Year"], keep=False)

if veg_dup.any():
    display(
        veg.loc[
            veg_dup,
            ["PlotID", "Year"],
        ].sort_values(["Year", "PlotID"])
    )
    raise ValueError(
        "Vegetation target contains duplicate PlotID-Year keys."
    )

print("Vegetation-state rows:", f"{len(veg):,}")

print("\nVegetation rows by year:")
display(
    veg.groupby("Year", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)


Vegetation-state rows: 1,171

Vegetation rows by year:


,Year,n
0,2016,365
1,2018,234
2,2025,118
3,2026,454


In [4]:
# =============================================================================
# SUPERVISED TARGET QA
# =============================================================================

parent_membership_cols = [
    f"membership_{i}"
    for i in range(1, 9)
]

parent_membership_sum = veg[parent_membership_cols].sum(axis=1)

print(
    "Parent membership row-sum range:",
    float(parent_membership_sum.min()),
    "to",
    float(parent_membership_sum.max()),
)

if not np.allclose(parent_membership_sum, 1.0, atol=1e-6):
    raise ValueError(
        "Parent fuzzy memberships do not sum to 1 within tolerance."
    )

print("\nParent-cluster counts:")
display(
    veg.groupby("parent_cluster", dropna=False)
    .size()
    .rename("n")
    .to_frame()
)

print("\nFinal state counts:")
display(
    veg.groupby(["state_type", "state_id"], dropna=False)
    .size()
    .rename("n")
    .to_frame()
)


Parent membership row-sum range: 0.9999999999999992 to 1.0000000000000002

Parent-cluster counts:


,n
parent_cluster,
1,178
2,139
3,128
4,220
5,146
6,129
7,124
8,107



Final state counts:


n
state_type       state_id     
nonvegetated     BG          4
vegetation_state 1.1       178
                 2.1        62
                 2.2        51
                 2.3        26
                 3.1        14
                 3.2        47
                 3.3        17
                 3.4        28
                 3.5        18
                 4.1       220
                 5.1        39
                 5.2        64
                 5.3        43
                 6.1        30
                 6.2        49
                 6.3        50
                 7.1       124
                 8.1        83
                 8.2        24

## 2. Load and validate the corrected positional master

This table is now the **single coordinate authority** for all years.


In [5]:
# =============================================================================
# POSITION MASTER
# =============================================================================

if not MASTER_POINT_CSV.exists():
    raise FileNotFoundError(MASTER_POINT_CSV)

points_raw = pd.read_csv(MASTER_POINT_CSV, low_memory=False)

plot_col = find_col(points_raw, ["PlotID", "Plot"])
year_col = find_col(points_raw, ["Year", "year"])
easting_col = find_col(points_raw, ["Easting", "EASTING"])
northing_col = find_col(points_raw, ["Northing", "NORTHING"])

coords = pd.DataFrame({
    "PlotID": normalize_plot_id(points_raw[plot_col]),
    "Year": pd.to_numeric(points_raw[year_col], errors="coerce"),
    "Easting": pd.to_numeric(points_raw[easting_col], errors="coerce"),
    "Northing": pd.to_numeric(points_raw[northing_col], errors="coerce"),
})

missing_key = coords["PlotID"].isna() | coords["Year"].isna()

if missing_key.any():
    print("Rows with missing PlotID or Year:", int(missing_key.sum()))
    display(coords.loc[missing_key].head(50))
    raise ValueError(
        "Position master contains missing PlotID/Year values."
    )

coords["Year"] = coords["Year"].astype(int)

missing_xy = coords["Easting"].isna() | coords["Northing"].isna()

if missing_xy.any():
    print("Rows with missing coordinates:", int(missing_xy.sum()))
    display(
        coords.loc[
            missing_xy,
            ["PlotID", "Year", "Easting", "Northing"],
        ].head(100)
    )
    raise ValueError(
        "Position master contains missing Easting/Northing values."
    )

# Intentionally broad UTM Zone 11N sanity bounds.
bad_xy = (
    ~coords["Easting"].between(100000, 900000)
    | ~coords["Northing"].between(4000000, 5500000)
)

if bad_xy.any():
    display(
        coords.loc[
            bad_xy,
            ["PlotID", "Year", "Easting", "Northing"],
        ].head(100)
    )
    raise ValueError(
        "Position master contains coordinates outside broad UTM 11N sanity bounds."
    )

dup = coords.duplicated(["PlotID", "Year"], keep=False)

if dup.any():
    print("Duplicate PlotID-Year keys remain in the position master:")
    display(
        coords.loc[dup]
        .sort_values(["Year", "PlotID"])
    )
    raise ValueError(
        "Resolve duplicate PlotID-Year keys before raster extraction."
    )

print("Position-master rows:", f"{len(coords):,}")

print("\nPosition rows by year:")
display(
    coords.groupby("Year", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)


Position-master rows: 1,179

Position rows by year:


,Year,n
0,2016,365
1,2018,231
2,2025,118
3,2026,465


## 3. Define supervised analysis years, then join vegetation to positions

Only vegetation years with a finalized annual K2 raster enter the current supervised experiment. This prevents 2026 from appearing as a coordinate-join failure when there is no 2026 raster to model against.

Extra points in the positional master are allowed. The supervised workflow is driven by the vegetation-state table.


In [6]:
# =============================================================================
# SUPERVISED ANALYSIS YEARS + VEGETATION ↔ POSITION JOIN
# =============================================================================

all_veg_years = sorted(
    veg["Year"]
    .dropna()
    .astype(int)
    .unique()
)

analysis_years = [
    year
    for year in all_veg_years
    if annual_k2_path(year).exists()
]

excluded_no_raster_years = [
    year
    for year in all_veg_years
    if year not in analysis_years
]

print("Vegetation years:", all_veg_years)
print("Years entering supervised experiment:", analysis_years)
print("Excluded because annual K2 raster is absent:", excluded_no_raster_years)

if not analysis_years:
    raise RuntimeError(
        "No vegetation monitoring years currently have annual K2 rasters."
    )

# This is the actual supervised-analysis population.
veg_model = (
    veg.loc[
        veg["Year"].isin(analysis_years)
    ]
    .copy()
    .reset_index(drop=True)
)

veg_coord = veg_model.merge(
    coords,
    on=["PlotID", "Year"],
    how="left",
    validate="one_to_one",
)

veg_coord["has_coordinates"] = (
    veg_coord["Easting"].notna()
    & veg_coord["Northing"].notna()
)

coord_summary = (
    veg_coord.groupby("Year", as_index=False)
    .agg(
        n_vegetation=("PlotID", "size"),
        n_with_coordinates=("has_coordinates", "sum"),
    )
)

coord_summary["n_missing_coordinates"] = (
    coord_summary["n_vegetation"]
    - coord_summary["n_with_coordinates"]
)

display(coord_summary)

missing_coords = (
    veg_coord.loc[
        ~veg_coord["has_coordinates"],
        ["PlotID", "Year"],
    ]
    .sort_values(["Year", "PlotID"])
)

if len(missing_coords) > 0:
    print(
        f"Supervised-year vegetation rows without coordinates: "
        f"{len(missing_coords):,}"
    )
    display(missing_coords)
else:
    print(
        "Every vegetation-state row in the current supervised years "
        "has a corrected spatial coordinate."
    )


Vegetation years: [np.int64(2016), np.int64(2018), np.int64(2025), np.int64(2026)]
Years entering supervised experiment: [np.int64(2016), np.int64(2018), np.int64(2025)]
Excluded because annual K2 raster is absent: [np.int64(2026)]


,Year,n_vegetation,n_with_coordinates,n_missing_coordinates
0,2016,365,362,3
1,2018,234,216,18
2,2025,118,118,0


Supervised-year vegetation rows without coordinates: 21


,PlotID,Year
360,406,2016
474,869,2016
494,889,2016
598,1000,2018
3,101,2018
5,102,2018
7,103,2018
9,104,2018
11,105,2018
13,106,2018


## 4. Confirm the annual K2 raster registry

The registry still reports every vegetation year, but only raster-backed years are passed downstream.


In [7]:
# =============================================================================
# RASTER REGISTRY
# =============================================================================

raster_registry = pd.DataFrame({
    "Year": all_veg_years,
    "raster_path": [
        annual_k2_path(y)
        for y in all_veg_years
    ],
})

raster_registry["raster_exists"] = (
    raster_registry["raster_path"].map(Path.exists)
)

display(raster_registry)

available_years = (
    raster_registry.loc[
        raster_registry["raster_exists"],
        "Year",
    ]
    .astype(int)
    .tolist()
)

missing_years = (
    raster_registry.loc[
        ~raster_registry["raster_exists"],
        "Year",
    ]
    .astype(int)
    .tolist()
)

print("Available monitoring years:", available_years)
print("Monitoring years without annual K2 TIFF:", missing_years)

assert available_years == analysis_years


,Year,raster_path,raster_exists
0,2016,A:\NCA_DATA\S2_Harmonics\2016\2016_K2_Intercep...,True
1,2018,A:\NCA_DATA\S2_Harmonics\2018\2018_K2_Intercep...,True
2,2025,A:\NCA_DATA\S2_Harmonics\2025\2025_K2_Intercep...,True
3,2026,A:\NCA_DATA\S2_Harmonics\2026\2026_K2_Intercep...,False


Available monitoring years: [2016, 2018, 2025]
Monitoring years without annual K2 TIFF: [2026]


In [8]:
# =============================================================================
# RASTER SCHEMA QA
# =============================================================================

reference_band_names = None
reference_ia = None
schema_rows = []

for year in available_years:

    path = annual_k2_path(year)

    with rasterio.open(path) as src:

        # ---------------------------------------------------------------------
        # CRS QA
        #
        # Avoid CRS.from_epsg(26911) because this environment currently has
        # conflicting PROJ database versions.
        #
        # The annual rasters already carry a NAD83 / UTM Zone 11N WKT CRS.
        # Validate that directly from the stored metadata.
        # ---------------------------------------------------------------------

        if src.crs is None:
            raise RuntimeError(
                f"{path.name} has no CRS."
            )

        crs_text = src.crs.to_string().upper()

        if not (
            "NAD83" in crs_text
            and "UTM ZONE 11N" in crs_text
        ):
            raise RuntimeError(
                f"{path.name} does not appear to be "
                "NAD83 / UTM Zone 11N.\n"
                f"Raster CRS:\n{src.crs}"
            )

        band_names = get_band_names(src)

        intercept, amplitude, ia = identify_ia_bands(
            band_names
        )

        schema_rows.append({

            "Year": year,

            "bands": src.count,

            # We have validated the stored WKT above.
            "crs": "NAD83 / UTM Zone 11N (EPSG:26911)",

            "width": src.width,

            "height": src.height,

            "nodata": src.nodata,

            "left": src.bounds.left,

            "bottom": src.bounds.bottom,

            "right": src.bounds.right,

            "top": src.bounds.top,

            "n_intercept": len(intercept),

            "n_amplitude": len(amplitude),

            "n_IA": len(ia),

        })

        if reference_band_names is None:

            reference_band_names = band_names

            reference_ia = ia

        elif band_names != reference_band_names:

            raise ValueError(
                f"{year} band schema/order differs from "
                "the first available annual raster."
            )


schema_df = pd.DataFrame(
    schema_rows
)

display(
    schema_df
)

IA_COLS = reference_ia

ALL_BAND_COLS = reference_band_names

print(
    "\nIA45 feature names:"
)

print(
    IA_COLS
)

Intercept bands: 15
Amplitude bands: 30
IA total:        45
Intercept bands: 15
Amplitude bands: 30
IA total:        45
Intercept bands: 15
Amplitude bands: 30
IA total:        45


,Year,bands,crs,width,height,nodata,left,bottom,right,top,n_intercept,n_amplitude,n_IA
0,2016,91,NAD83 / UTM Zone 11N (EPSG:26911),10435,7857,-32768.0,527740.0,4736840.0,632090.0,4815410.0,15,30,45
1,2018,91,NAD83 / UTM Zone 11N (EPSG:26911),10435,7857,-32768.0,527740.0,4736840.0,632090.0,4815410.0,15,30,45
2,2025,91,NAD83 / UTM Zone 11N (EPSG:26911),10435,7857,-32768.0,527740.0,4736840.0,632090.0,4815410.0,15,30,45



IA45 feature names:
['constant_B2', 'constant_B3', 'constant_B4', 'constant_B5', 'constant_B6', 'constant_B7', 'constant_B8', 'constant_B8A', 'constant_B11', 'constant_B12', 'constant_NDVI', 'constant_NDMI', 'constant_NDRE', 'constant_BSI', 'constant_NBR2', 'amp1_B2', 'amp1_B3', 'amp1_B4', 'amp1_B5', 'amp1_B6', 'amp1_B7', 'amp1_B8', 'amp1_B8A', 'amp1_B11', 'amp1_B12', 'amp1_NDVI', 'amp1_NDMI', 'amp1_NDRE', 'amp1_BSI', 'amp1_NBR2', 'amp2_B2', 'amp2_B3', 'amp2_B4', 'amp2_B5', 'amp2_B6', 'amp2_B7', 'amp2_B8', 'amp2_B8A', 'amp2_B11', 'amp2_B12', 'amp2_NDVI', 'amp2_NDMI', 'amp2_NDRE', 'amp2_BSI', 'amp2_NBR2']


## 5. Sample each plot from the K2 raster corresponding to its monitoring year


In [9]:
# =============================================================================
# 5. MONITORING-YEAR RASTER EXTRACTION
# =============================================================================

def sample_year_raster(raster_path, point_df):
    """
    Sample every raster band at EPSG:26911 plot coordinates.

    Points outside raster bounds are explicitly retained with:
      in_raster_bounds = False
      all raster bands = NaN
    """

    point_df = point_df.copy().reset_index(drop=True)

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise RuntimeError(
                f"{raster_path} has no CRS."
            )

        # Compare CRS definitions, not their string representations.
        crs_text = src.crs.to_string().upper()

        if not (
            "NAD83" in crs_text
            and "UTM ZONE 11N" in crs_text
        ):
            raise RuntimeError(
        f"{raster_path.name} does not appear to be "
        "NAD83 / UTM Zone 11N.\n"
        f"Raster CRS:\n{src.crs}"
        )
        band_names = get_band_names(src)

        left, bottom, right, top = src.bounds

        point_df["in_raster_bounds"] = (
            point_df["Easting"].between(left, right, inclusive="both")
            & point_df["Northing"].between(bottom, top, inclusive="both")
        )

        # Preallocate all rows as NaN.
        sampled = pd.DataFrame(
            np.nan,
            index=np.arange(len(point_df)),
            columns=band_names,
            dtype=np.float32,
        )

        inside_idx = point_df.index[
            point_df["in_raster_bounds"]
        ].tolist()

        if inside_idx:

            inside_coords = list(
                zip(
                    point_df.loc[
                        inside_idx,
                        "Easting"
                    ].astype(float),
                    point_df.loc[
                        inside_idx,
                        "Northing"
                    ].astype(float),
                )
            )

            values = np.asarray(
                list(
                    src.sample(
                        inside_coords,
                        indexes=list(
                            range(1, src.count + 1)
                        ),
                        masked=False,
                    )
                ),
                dtype=np.float32,
            )

            inside_sampled = pd.DataFrame(
                values,
                columns=band_names,
                index=inside_idx,
            )

            if src.nodata is not None:
                inside_sampled = inside_sampled.replace(
                    src.nodata,
                    np.nan,
                )

            inside_sampled = inside_sampled.replace(
                [np.inf, -np.inf],
                np.nan,
            )

            sampled.loc[
                inside_idx,
                band_names
            ] = inside_sampled

    sampled.insert(
        0,
        "PlotID",
        point_df["PlotID"].to_numpy(),
    )

    sampled.insert(
        1,
        "Year",
        point_df["Year"].to_numpy(),
    )

    sampled.insert(
        2,
        "Easting",
        point_df["Easting"].to_numpy(),
    )

    sampled.insert(
        3,
        "Northing",
        point_df["Northing"].to_numpy(),
    )

    sampled.insert(
        4,
        "in_raster_bounds",
        point_df["in_raster_bounds"].to_numpy(),
    )

    return sampled


# =============================================================================
# SAMPLE EACH MONITORING YEAR
# =============================================================================

sampled_tables = []

for year in available_years:

    year_points = (
        veg_coord.loc[
            (veg_coord["Year"] == year)
            & veg_coord["has_coordinates"],
            [
                "PlotID",
                "Year",
                "Easting",
                "Northing",
            ],
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(year_points) == 0:
        print(
            f"{year}: no vegetation rows with coordinates; skipping."
        )
        continue

    path = annual_k2_path(year)

    print(
        f"{year}: evaluating {len(year_points):,} vegetation plots "
        f"against {path.name}"
    )

    year_sampled = sample_year_raster(
        path,
        year_points,
    )

    print(
        "  inside raster bounds:",
        f"{int(year_sampled['in_raster_bounds'].sum()):,}",
    )

    print(
        "  outside raster bounds:",
        f"{int((~year_sampled['in_raster_bounds']).sum()):,}",
    )

    sampled_tables.append(
        year_sampled
    )


if not sampled_tables:
    raise RuntimeError(
        "No raster samples were produced."
    )


sampled = pd.concat(
    sampled_tables,
    ignore_index=True,
)


print(
    "\nSampled/audited rows:",
    f"{len(sampled):,}",
)


display(
    sampled.groupby(
        "Year",
        as_index=False,
    )
    .agg(
        n_evaluated=(
            "PlotID",
            "size",
        ),
        n_in_raster_bounds=(
            "in_raster_bounds",
            "sum",
        ),
    )
)

2016: evaluating 362 vegetation plots against 2016_K2_InterceptAmpPhase_nobs_RMSE.tif
  inside raster bounds: 257
  outside raster bounds: 105
2018: evaluating 216 vegetation plots against 2018_K2_InterceptAmpPhase_nobs_RMSE.tif
  inside raster bounds: 114
  outside raster bounds: 102
2025: evaluating 118 vegetation plots against 2025_K2_InterceptAmpPhase_nobs_RMSE.tif
  inside raster bounds: 118
  outside raster bounds: 0

Sampled/audited rows: 696


,Year,n_evaluated,n_in_raster_bounds
0,2016,362,257
1,2018,216,114
2,2025,118,118


## 6. Join optical samples back to vegetation states and diagnose attrition

Because non-raster years were excluded before the coordinate join, current attrition now has interpretable causes:

- missing coordinate match in a raster-backed monitoring year
- coordinate outside the annual raster extent
- in-bounds point with incomplete/masked IA predictors
- usable IA45 sample


In [10]:
# =============================================================================
# JOIN OPTICAL FEATURES BACK TO VEGETATION TARGETS
# =============================================================================

sampled_features = sampled.drop(
    columns=[
        "Easting",
        "Northing",
    ],
    errors="ignore",
)

training_all = veg_coord.merge(
    sampled_features,
    on=["PlotID", "Year"],
    how="left",
    validate="one_to_one",
)

available_set = set(available_years)

training_all["has_year_raster"] = (
    training_all["Year"].isin(available_set)
)

training_all["in_raster_bounds"] = (
    training_all["in_raster_bounds"]
    .fillna(False)
    .astype(bool)
)

ia_numeric = (
    training_all[IA_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

training_all["ia_complete"] = (
    ia_numeric.notna().all(axis=1)
)

training_all["n_ia_missing"] = (
    ia_numeric.isna().sum(axis=1)
)


def extraction_status(row):
    if not row["has_coordinates"]:
        return "missing_coordinates"
    if not row["has_year_raster"]:
        return "missing_year_raster"
    if not row["in_raster_bounds"]:
        return "outside_raster_extent"
    if not row["ia_complete"]:
        return "incomplete_or_masked_IA"
    return "usable_IA45"


training_all["extraction_status"] = training_all.apply(
    extraction_status,
    axis=1,
)

display(
    training_all.groupby(
        ["Year", "extraction_status"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n"})
)


,Year,extraction_status,n
0,2016,incomplete_or_masked_IA,2
1,2016,missing_coordinates,3
2,2016,outside_raster_extent,105
3,2016,usable_IA45,255
4,2018,incomplete_or_masked_IA,8
5,2018,missing_coordinates,18
6,2018,outside_raster_extent,102
7,2018,usable_IA45,106
8,2025,usable_IA45,118


In [11]:
# =============================================================================
# OPTIONAL QA BANDS
# =============================================================================

nobs_cols = [
    c for c in ALL_BAND_COLS
    if (
        re.fullmatch(r"nobs", c, flags=re.IGNORECASE)
        or re.search(r"^nobs_", c, flags=re.IGNORECASE)
    )
]

rmse_cols = [
    c for c in ALL_BAND_COLS
    if re.search(r"rmse", c, flags=re.IGNORECASE)
]

print("nobs columns:", nobs_cols)
print("RMSE columns:", len(rmse_cols))

if nobs_cols:
    nobs_col = nobs_cols[0]

    display(
        training_all.loc[
            training_all[nobs_col].notna()
        ]
        .groupby("Year")[nobs_col]
        .describe()
        .round(2)
    )


nobs columns: ['nobs']
RMSE columns: 15


,count,mean,std,min,25%,50%,75%,max
Year,,,,,,,,
2016,255.0,26.60,5.03,12.0,26.00,28.0,30.0,35.0
2018,106.0,58.86,14.01,22.0,40.25,66.0,69.0,72.0
2025,118.0,80.14,13.81,45.0,83.00,85.0,87.0,91.0


## 7. Construct parent-cluster and subcluster supervised tables in parallel

Both tables use the **same usable plot-years and the same IA45 predictors**. The only substantive difference is response resolution.

This keeps the modeling question open:

- Can Sentinel-2 recover broad parent vegetation states?
- Can it recover the finer nested subclusters?


In [12]:
# =============================================================================
# PARALLEL SUPERVISED TABLES
# =============================================================================

usable = (
    training_all.loc[
        training_all["extraction_status"].eq("usable_IA45")
    ]
    .copy()
    .reset_index(drop=True)
)

id_cols = [
    "PlotID",
    "Year",
    "Easting",
    "Northing",
]

qa_cols = [
    c for c in nobs_cols
    if c in usable.columns
]

# -----------------------------------------------------------------------------
# A. PARENT-CLUSTER RESPONSE
# -----------------------------------------------------------------------------

parent_response_cols = [
    "parent_cluster",
    "parent_label",
    "max_membership",
    "second_membership",
    "membership_margin",
    *[f"membership_{i}" for i in range(1, 9)],
]

parent_response_cols = [
    c for c in parent_response_cols
    if c in usable.columns
]

parent_cols = []

for c in (
    id_cols
    + parent_response_cols
    + qa_cols
    + IA_COLS
):
    if (
        c in usable.columns
        and c not in parent_cols
    ):
        parent_cols.append(c)

training_parent = (
    usable.loc[
        usable["parent_cluster"].notna(),
        parent_cols,
    ]
    .copy()
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# B. SUBCLUSTER / FINAL-STATE RESPONSE
# -----------------------------------------------------------------------------

subcluster_response_cols = [
    # retain parent context while modeling the finer response
    "parent_cluster",
    "parent_label",
    "raw_subcluster",
    "state_id",
    "state_label",
    "state_type",
    "species_centroid_distance",
    "is_reference",
    # retain parent uncertainty diagnostics for interpretation
    "max_membership",
    "membership_margin",
]

subcluster_response_cols = [
    c for c in subcluster_response_cols
    if c in usable.columns
]

subcluster_cols = []

for c in (
    id_cols
    + subcluster_response_cols
    + qa_cols
    + IA_COLS
):
    if (
        c in usable.columns
        and c not in subcluster_cols
    ):
        subcluster_cols.append(c)

training_subcluster = (
    usable.loc[
        usable["state_id"].notna(),
        subcluster_cols,
    ]
    .copy()
    .reset_index(drop=True)
)

training_subcluster["state_id"] = (
    training_subcluster["state_id"].astype("string")
)


# -----------------------------------------------------------------------------
# SUMMARY
# -----------------------------------------------------------------------------

print("IA45 predictors:", len(IA_COLS))
print("Usable optical plot-years:", f"{len(usable):,}")
print("Parent-cluster training rows:", f"{len(training_parent):,}")
print("Subcluster training rows:", f"{len(training_subcluster):,}")

print("\nPARENT DATASET BY YEAR")
display(
    training_parent.groupby("Year", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

print("\nSUBCLUSTER DATASET BY YEAR")
display(
    training_subcluster.groupby("Year", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

print("\nPARENT-CLUSTER COUNTS")
display(
    training_parent.groupby(
        "parent_cluster",
        dropna=False,
    )
    .size()
    .rename("n")
    .sort_values(ascending=False)
    .to_frame()
)

print("\nSUBCLUSTER / STATE COUNTS")
display(
    training_subcluster.groupby(
        "state_id",
        dropna=False,
    )
    .size()
    .rename("n")
    .sort_values(ascending=False)
    .to_frame()
)


IA45 predictors: 45
Usable optical plot-years: 479
Parent-cluster training rows: 479
Subcluster training rows: 479

PARENT DATASET BY YEAR


,Year,n
0,2016,255
1,2018,106
2,2025,118



SUBCLUSTER DATASET BY YEAR


,Year,n
0,2016,255
1,2018,106
2,2025,118



PARENT-CLUSTER COUNTS


,n
parent_cluster,
1,83
4,82
3,78
5,74
2,59
6,51
7,32
8,20



SUBCLUSTER / STATE COUNTS


,n
state_id,
1.1,83
4.1,82
5.2,48
7.1,32
3.2,31
6.2,26
2.2,26
2.1,19
8.1,16


## 8. Year-level attrition audit

This now describes only raster-backed supervised years, so attrition is not contaminated by 2026 or other years lacking an annual K2 product.


In [13]:
# =============================================================================
# YEAR-LEVEL ATTRITION SUMMARY
# =============================================================================

year_summary = (
    training_all.groupby("Year", as_index=False)
    .agg(
        n_vegetation=("PlotID", "size"),
        n_with_coordinates=("has_coordinates", "sum"),
        n_with_year_raster=("has_year_raster", "sum"),
        n_in_raster_bounds=("in_raster_bounds", "sum"),
        n_IA45_complete=("ia_complete", "sum"),
    )
)

year_summary["lost_coordinate"] = (
    year_summary["n_vegetation"]
    - year_summary["n_with_coordinates"]
)

year_summary["lost_no_year_raster"] = (
    year_summary["n_with_coordinates"]
    - year_summary["n_with_year_raster"]
)

year_summary["lost_outside_raster_extent"] = (
    year_summary["n_with_year_raster"]
    - year_summary["n_in_raster_bounds"]
)

year_summary["lost_in_bounds_mask_or_incomplete_IA"] = (
    year_summary["n_in_raster_bounds"]
    - year_summary["n_IA45_complete"]
)

year_summary["retention_fraction"] = (
    year_summary["n_IA45_complete"]
    / year_summary["n_vegetation"]
)

display(year_summary)


,Year,n_vegetation,n_with_coordinates,n_with_year_raster,n_in_raster_bounds,n_IA45_complete,lost_coordinate,lost_no_year_raster,lost_outside_raster_extent,lost_in_bounds_mask_or_incomplete_IA,retention_fraction
0,2016,365,362,365,257,255,3,-3,108,2,0.698630
1,2018,234,216,234,114,106,18,-18,120,8,0.452991
2,2025,118,118,118,118,118,0,0,0,0,1.000000


## 9. Write full audit plus both supervised response tables


In [14]:
# =============================================================================
# OUTPUTS
# =============================================================================

training_all.to_csv(
    OUT_ALL,
    index=False,
)

training_parent.to_csv(
    OUT_PARENT,
    index=False,
)

training_subcluster.to_csv(
    OUT_SUBCLUSTER,
    index=False,
)

audit_cols = [
    c for c in [
        "PlotID",
        "Year",
        "Easting",
        "Northing",
        "has_coordinates",
        "has_year_raster",
        "in_raster_bounds",
        "ia_complete",
        "n_ia_missing",
        "extraction_status",
        "parent_cluster",
        "max_membership",
        "membership_margin",
        "state_id",
        "state_label",
        "state_type",
    ]
    if c in training_all.columns
]

training_all[audit_cols].to_csv(
    OUT_AUDIT,
    index=False,
)

year_summary.to_csv(
    OUT_YEAR_SUMMARY,
    index=False,
)

print("WROTE:")

for path in [
    OUT_ALL,
    OUT_PARENT,
    OUT_SUBCLUSTER,
    OUT_AUDIT,
    OUT_YEAR_SUMMARY,
]:
    print(" ", path)


WROTE:
  A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_AllSampledBands.csv
  A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_IA45_ParentClusters.csv
  A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_IA45_Subclusters.csv
  A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_ExtractionAudit.csv
  A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_YearSummary.csv


## 10. Final structural QA for both response resolutions


In [15]:
# =============================================================================
# FINAL QA
# =============================================================================

assert len(IA_COLS) == 45

for name, df, target in [
    ("parent", training_parent, "parent_cluster"),
    ("subcluster", training_subcluster, "state_id"),
]:

    assert df[target].notna().all()

    assert (
        df[IA_COLS]
        .notna()
        .all()
        .all()
    )

    assert np.isfinite(
        df[IA_COLS].to_numpy(dtype=float)
    ).all()

    assert not df.duplicated(
        ["PlotID", "Year"]
    ).any()

    print(
        f"{name}:",
        f"{len(df):,} rows,",
        f"{df[target].nunique()} response classes"
    )

# With complete vegetation labels, these should normally be identical.
same_plot_years = (
    set(
        map(
            tuple,
            training_parent[["PlotID", "Year"]].to_numpy()
        )
    )
    ==
    set(
        map(
            tuple,
            training_subcluster[["PlotID", "Year"]].to_numpy()
        )
    )
)

print(
    "\nParent and subcluster tables use identical plot-years:",
    same_plot_years,
)

print(
    "Current supervised years:",
    sorted(training_parent["Year"].unique().tolist())
)

print(
    "Excluded vegetation years with no annual raster:",
    excluded_no_raster_years,
)

print("\nParallel supervised datasets complete.")


parent: 479 rows, 8 response classes
subcluster: 479 rows, 20 response classes

Parent and subcluster tables use identical plot-years: True
Current supervised years: [2016, 2018, 2025]
Excluded vegetation years with no annual raster: [np.int64(2026)]

Parallel supervised datasets complete.
